# Exercises: Open Data Structures - Chapter 1

## Exercise 1.1. 

This exercise is designed to help familiarize the reader with
choosing the right data structure for the right problem. If implemented, the parts of this exercise should be done by making use of an implemen tation of the relevant interface (Stack, Queue, Deque, USet, or SSet) provided by the .  

Solve the following problems by reading a text file one line at a time and performing operations on each line in the appropriate data structure(s). Your implementations should be fast enough that even files containing a million lines can be processed in a few seconds.


In [46]:
# create file with numbered lines with random words...

import os
from pathlib import Path
import random


root = Path(os.path.abspath(os.sep))
words_path = root / "usr" / "share" / "dict" / "words"

# create data:
def random_words(n):
    with open(words_path) as f:
        words = f.read().split()
    return " ".join(random.choice(words) for _ in range(n))


with open("random_lines.txt", mode="w", encoding="utf-8") as f:
    for i in range(1, 201):
        f.writelines(f"line_{str(i).zfill(3)}: {random_words(2)}\n")


### 1.1.1 Read the input one line at a time 

and then write the lines out in reverse order, so that the last input line is printed first, then the second last input line, and so on.

In [47]:
class UserStack:

    def __init__(self):
        self.stack = []

    def push(self, obj):
        self.stack.append(obj)

    def pop(self):
        return self.stack.pop()

In [48]:
stack = UserStack()
with open("random_lines.txt", mode="r", encoding="utf-8") as f:
    for line in f.readlines():
        stack.push(line)

In [49]:
with open("reversed_rl.txt", mode="w", encoding="utf-8") as f:
    while stack.stack:
        f.writelines(stack.pop())

### 1.1.2. reverse 50 at a time 

Read the first 50 lines of input and then write them out in reverse order. Read the next 50 lines and then write them out in reverse order. Do this until there are no more lines left to read, at which point any remaining lines should be output in reverse order.  

In other words, your output will start with the 50th line, then the 49th, then the 48th, and so on down to the first line. This will be followed by the 100th line, followed by the 99th, and so on down to the 51st line. And so on.  

Your code should never have to store more than 50 lines at any given time.

In [50]:
# this is also a stack problem.
# we will use the same stack,
# but push and pop according to
# batch_size (50).

def reverse_batch(in_path:Path|str, out_path:Path|str, batch_size:int) -> None:
    stack = UserStack()

    with open(out_path, mode="w") as out:
        out.write(f"--- Reversing in batch {batch_size} ---\n")

    with open(in_path, mode="r", encoding="utf-8") as input:
        for line in input:
            stack.push(line)

            if len(stack.stack) == batch_size:
                with open(out_path, mode="a", encoding="utf-8") as out:
                    while stack.stack:
                        out.writelines(stack.pop())
                    out.write("-" * 50 + "\n")

        while stack.stack:
            with open(out_path, mode="a", encoding="utf-8") as out:
                out.writelines(stack.pop())

In [51]:
in_path = Path("random_lines.txt")

reverse_batch(in_path, "reverse_50.txt", 50)
reverse_batch(in_path, "reverse_30.txt", 30)

### 1.1.3. Output n before blankline

Read the input one line at a time. At any point after reading the first 42 lines, if some line is blank (i.e., a string of length 0), then output the line that occured 42 lines prior to that one. 

For example, if Line 242 is blank, then your program should output line 200. This program should be implemented so that it never stores more than 43 lines of the input at any given time.

In [52]:
# we will use the same structure as a Queue system, FIFO,
# but I will not bother creating a new custom datastructure.
# we will use list as a sliding window of sorts,
# pop at index 0, and appending, while striding
# with instructed size.

def get_n_before_blankline(input_path:Path|str, n:int=42) -> str:
    with open(input_path, mode="r", encoding="utf-8") as f:
        # initialize with n number of lines
        lines = [f.readline() for _ in range(n)]
        while True:
            line = f.readline()
            if not line:
                return None
            elif line == "\n":
                return lines[0].removesuffix("\n")
            else:
                lines.pop(0)
                lines.append(line)


get_n_before_blankline("w_blanklines.txt")

"line_100: Swahili's compatibles"

In [53]:
# deque is faster because it performs append and discard operations in O(1) time.
# A normal list must shift all elements when you pop from the front, which is O(n).
# deque avoids this by using a double-ended buffer structure that never needs to
# move existing elements in memory.

# A while-loop is not required because a file object is already an iterator.
# Using "for line in f" reads one line at a time until EOF, giving the same behavior
# as a manual while-loop with readline(), but with cleaner control flow and no
# explicit break conditions.

from collections import deque
from pathlib import Path

def get_n_before_blankline(input_path: str | Path, n: int = 42) -> str | None:
    window = deque(maxlen=n)

    with open(input_path, "r", encoding="utf-8") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                return None
            window.append(line)

        for line in f:
            if line == "\n":
                return window[0].rstrip("\n")
            window.append(line)

    return None

get_n_before_blankline("w_blanklines.txt")

"line_100: Swahili's compatibles"

### 1.1.4. 

Read the input one line at a time and write each line to the output
if it is not a duplicate of some previous input line. Take special care
so that a file with a lot of duplicate lines does not use more memory
than what is required for the number of unique lines.

In [54]:
# The correct memory-efficient solution is to store only unique lines in a set.
# This satisfies the requirement: memory usage grows only with the number of
# distinct lines, never with the total number of lines in the file.

# Using a set avoids scanning the output file repeatedly, which would be extremely
# slow because it requires reading the entire file for every new input line.
# A set provides O(1) average-time membership checks, making it ideal for detecting
# duplicates without loading the whole input into memory.

# The assignment does NOT forbid storing all unique lines; it only forbids storing
# all lines. Therefore, keeping a set of unique lines is exactly the intended solution.

def copy_wo_duplicates(input_path: Path | str, out_path: Path | str):

    unique_lines = set()
    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITHOUT DUPLICATES  ===\n")

    with open(input_path, mode="r", encoding="utf-8") as input:
        with open(out_path, mode="a", encoding="utf-8") as output:
            for line in input:
                if line not in unique_lines:
                    unique_lines.add(line)
                    output.write(line)

copy_wo_duplicates("w_duplicates.txt", "no_duplicates.txt")

### 1.1.5. 

Read the input one line at a time and write each line to the output
only if you have already read this line before. The end result is that you remove the first occurrence of each line. Take special care so that a file with a lot of duplicate lines does not use more memory than what is required for the number of unique lines.

In [55]:
def copy_wo_duplicates(input_path: Path | str, out_path: Path | str):
    unique_lines = set()
    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITH ONLY DUPLICATES  ===\n")

    with open(input_path, mode="r", encoding="utf-8") as input:
        with open(out_path, mode="a", encoding="utf-8") as output:
            for line in input:
                if line in unique_lines:
                    output.write(line)
                unique_lines.add(line)

copy_wo_duplicates("w_duplicates.txt", "only_duplicates.txt")

### 1.1.6. 

Read the entire input one line at a time. Then output all lines sorted by length, with the shortest lines first.  In the case where two lines have the same length, resolve their order using the usual “sorted order.” Duplicate lines should be printed only once.

In [56]:
# We use the same strategies as in 1.1.4 and 1.1.5, and first store
# uniques in a set.
# then we cast to list, use sorted with a tuple key on length + alpha

def sort_uniques_by_length(input_path: Path | str, out_path: Path | str):
    unique_lines = set()

    with open(input_path, mode="r", encoding="utf-8") as input:
        for line in input:
            unique_lines.add(line)

    prio_que = sorted(list(unique_lines), key=lambda line : (len(line), line))
    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITH LINES SORTED BY LENGTH ===\n")
        for line in prio_que:
            output.write(line)

sort_uniques_by_length("w_duplicates.txt", "uniques_sorted_len.txt")

### 1.1.7. 

Do the same as the previous question except that duplicate lines should be printed the same number of times that they appear in the input.

In [ ]:
def sort_by_length(input_path: Path | str, out_path: Path | str):
    with open(input_path, mode="r", encoding="utf-8") as input:
        lines = input.readlines()

    prio_que = sorted(lines, key=lambda line : (len(line), line))
    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITH LINES SORTED BY LENGTH ===\n")
        for line in prio_que:
            output.write(line)

sort_by_length("w_duplicates.txt", "sorted_len.txt")

[]


### 1.1.8. 

Read the entire input one line at a time and then output the even numbered lines (starting with the first line, line 0) followed by the odd-numbered lines.

In [58]:
def shuffle(input_path: Path | str, out_path: Path | str):
    even = []
    odd = []
    with open(input_path, mode="r", encoding="utf-8") as input:
        for i, line in enumerate(input, start=1):
            if i % 2 == 0:
                even.append(line)
            else:
                odd.append(line)

    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITH SHUFFLED LINES (even, odd) ===\n")
        for line in even:
            output.write(line)
        for line in odd:
            output.write(line)

shuffle("random_lines.txt", "shuffled.txt")

### 1.1.9. 

Read the entire input one line at a time and randomly permute the
lines before outputting them. To be clear: You should not modify
the contents of any line. Instead, the same collection of lines should 
be printed, but in a random order.

In [59]:
import random

def permute(input_path: Path | str, out_path: Path | str):
    with open(input_path, mode="r", encoding="utf-8") as input:
        lines = input.readlines()

    with open(out_path, mode="w", encoding="utf-8") as output:
        output.write("=== FILE WITH SHUFFLED LINES (even, odd) ===\n")

        while lines:
            idx = random.randrange(0, len(lines))
            output.write(lines[idx])
            lines.pop(idx)

permute("random_lines.txt", "permuted.txt")

## Exercise 1.2. 

A Dyck word is a sequence of +1’s and -1’s with the property
that the sum of any prefix of the sequence is never negative. 

For example, +1, −1, +1, −1 is a Dyck word, but +1, −1, −1, +1 is not a Dyck word since
the prefix +1 − 1 − 1 < 0. 

Describe any relationship between Dyck words and Stack push(x) and pop() operations.

In [60]:
# - Dyckwords stay on the surface, always picking from top of stack
# - non-Dyckwords digs deeper

## Exercise 1.3. 

A matched string is a sequence of {, }, (, ), [, and ] characters
that are properly matched. For example, “{{()[]}}” is a matched string, but
this “{{()]}” is not, since the second { is matched with a ]. Show how to
use a stack so that, given a string of length n, you can determine if it is a
matched string in O(n) time.

In [61]:
# first, an example that proves why
# a stack is needed.

def is_matched_string(s:str) -> bool:
    curls = 0
    parents = 0
    clams = 0

    for c in s:
        match c:
            case "{": curls += 1
            case "}": curls -= 1
            case "(": parents += 1
            case ")": parents -= 1
            case "[": clams += 1
            case "]": clams -= 1

    if any([curls, parents, clams]):
        return False
    return True

test_string = "{([])}"
print(test_string, is_matched_string(test_string))

test_string = "{([})}"
print(test_string, is_matched_string(test_string))

test_string = "{(wrong[})order]"
print(test_string, is_matched_string(test_string))

{([])} True
{([})} False
{(wrong[})order] True


In [62]:
class UserStack:
    def __init__(self):
        self.stack = []

    def push(self, obj):
        self.stack.append(obj)

    def pop(self):
        return self.stack.pop()

In [63]:
def is_matched_string(s:str) -> bool:
    stack = UserStack()
    for c in s:
        if c in ["{", "(", "["]:
            stack.push(c)
        elif c in ["}", ")", "]"]:
            if stack.pop() + c not in [r"{}", "()", "[]"]:
                return False
    return True


test_string = "{zzzzzzzz([asdfasdf]hej)}"
print(test_string, is_matched_string(test_string))

test_string = "{([})}"
print(test_string, is_matched_string(test_string))


{zzzzzzzz([asdfasdf]hej)} True
{([})} False


## Exercise 1.4. 

Suppose you have a Stack, **s**, that supports only the push(x)
and pop() operations. Show how, using only a FIFO Queue, **q**, you can
reverse the order of all elements in **s**

In [89]:
import queue

def reverse_stack(stack:UserStack) -> None:
    fifo_queue = queue.Queue()
    while stack.stack:
        fifo_queue.put(stack.pop())

    while not fifo_queue.empty():
        stack.push(fifo_queue.get())

In [90]:
my_stack = UserStack()
for c in "abcdefg":
    my_stack.push(c)

print(my_stack.stack)
reverse_stack(my_stack)
print(my_stack.stack)

['a', 'b', 'c', 'd', 'e', 'f', 'g']
['g', 'f', 'e', 'd', 'c', 'b', 'a']


# Exercise 1.5. 

Using a USet, implement a Bag. A Bag is like a USet (Unordered Set) — it supports the 
add(x), remove(x) and find(x) methods—but it allows duplicate
elements to be stored. 

The find(x) operation in a Bag returns some 
element (if any) that is equal to x. 

In addition, a Bag supports the find all(x)
operation that returns a list of all elements in the Bag that are equal to x.

---

| Implementation      | find(x)                      | add(x)/remove(x)                    | Section |
|---------------------|------------------------------|-------------------------------------|---------|
| ChainedHashTable    | O(1)^E                       | O(1)^A,E                            | § 5.1   |
| LinearHashTable     | O(1)^E                       | O(1)^A,E                            | § 5.2   |


^A Denotes an amortized running time.  
^E Denotes an expected running time.


In [105]:
class BagItem:
    def __init__(self, x):
        self.x = x
        self.count = 1

    def __eq__(self, other):
        # Treat BagItem as equal to the raw value 'other'
        return self.x == other


class USet:
    def __init__(self):
        self.items = []  # unordered set of unique objects

    def size(self):
        return len(self.items)

    def add(self, x):
        # find(x) returns the actual stored object if equal
        if self.find(x) is not None:
            return False
        self.items.append(x)
        return True

    def remove(self, x):
        # find the stored object y such that y == x
        for i, y in enumerate(self.items):
            if y == x:
                self.items.pop(i)
                return y
        return None

    def find(self, x):
        # return the stored object y such that y == x
        for y in self.items:
            if y == x:
                return y
        return None


class Bag:
    def __init__(self):
        self.uset = USet()  # abstract set storing BagItem objects

    def add(self, x):
        item = self.uset.find(x)
        if item is None:
            self.uset.add(BagItem(x))
        else:
            item.count += 1

    def remove(self, x):
        item = self.uset.find(x)
        if item is None:
            return
        item.count -= 1
        if item.count == 0:
            self.uset.remove(item)

    def count(self, x):
        item = self.uset.find(x)
        return 0 if item is None else item.count


In [111]:
my_bag = Bag()

my_bag.add("a")
my_bag.add("a")
my_bag.add("a")

print(my_bag.count("a"))

my_bag.remove("a")
print(my_bag.count("a"))

3
2


In [113]:
bag = {}

bag["a"] = bag.get("a", 0) + 1

bag["a"]

1